In [ ]:
#ANN Regression
import pandas as pd


In [54]:
df=pd.read_csv("10.1_DateFruit_Dataset.csv")
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [55]:
x=df.drop("Class",axis=1)
y=df["Class"]


In [56]:
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()
y=le.fit_transform(y)

In [57]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(
    x,y,test_size=0.2,random_state=42)

In [58]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)


In [59]:
import torch
import torch.nn as nn

In [60]:
x_train_tensor=torch.tensor(x_train_scaled,dtype=torch.float32)
x_test_tensor=torch.tensor(x_test_scaled,dtype=torch.float32)


y_train_tensor=torch.tensor(y_train,dtype=torch.long)
y_test_tensor=torch.tensor(y_test,dtype=torch.long)

In [61]:
from torch.utils.data import TensorDataset,DataLoader
train_data=TensorDataset(x_train_tensor,y_train_tensor)
test_data=TensorDataset(x_test_tensor,y_test_tensor)

In [62]:
train_loader=DataLoader(train_data,batch_size=32,shuffle=True)
test_loader=DataLoader(test_data,batch_size=32)

In [63]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()
        self.model=nn.Sequential(

            nn.Linear(x.shape[1],64),
            nn.ReLU(),

            nn.Linear(64,64),
            nn.ReLU(),

            nn.Linear(64,7)
        )
    def forward(self,x):
        return self.model(x)
            
            
        

In [64]:
import torch.optim as optim
model=ANN()
criteria=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [67]:
epochs=100
running_loss=0.0
for epoch in range(epochs):
    model.train()
    for xb,yb in train_loader:
        optimizer.zero_grad()
        outputs=model(xb)
        loss=criteria(outputs,yb)
        loss.backward()
        optimizer.step()

        running_loss+=loss.item()

    train_loss=running_loss/len(train_loader)
    print(f" epoch={epoch+1}/{epochs},loss={train_loss}")

    


 epoch=1/100,loss=0.03275818823148375
 epoch=2/100,loss=0.05822847874673164
 epoch=3/100,loss=0.08538867051348738
 epoch=4/100,loss=0.11291466890226888
 epoch=5/100,loss=0.13662508839700857
 epoch=6/100,loss=0.16086210140391535
 epoch=7/100,loss=0.186064610855006
 epoch=8/100,loss=0.20939582548833088
 epoch=9/100,loss=0.24452673016197007
 epoch=10/100,loss=0.267212550411426
 epoch=11/100,loss=0.2886509084998144
 epoch=12/100,loss=0.31116005189403
 epoch=13/100,loss=0.33433654183349776
 epoch=14/100,loss=0.35464342916131264
 epoch=15/100,loss=0.3745616068380237
 epoch=16/100,loss=0.39319471061563765
 epoch=17/100,loss=0.4123189676193642
 epoch=18/100,loss=0.4307531698672708
 epoch=19/100,loss=0.4486972949842153
 epoch=20/100,loss=0.4663972162803554
 epoch=21/100,loss=0.4845840498278646
 epoch=22/100,loss=0.5044901174108457
 epoch=23/100,loss=0.5214547979817523
 epoch=24/100,loss=0.536712766680664
 epoch=25/100,loss=0.5529422586550936
 epoch=26/100,loss=0.5716153752468729
 epoch=27/100,l

In [68]:
model.eval()
total=0
correct=0

with torch.no_grad():
    for xb,yb in test_loader:
        outputs=model(xb)
        _,predicted=torch.max(outputs,1)

        correct+=(predicted==yb).sum().item()
        total+=yb.size(0)

print("accuracy:",correct/total *100)

accuracy: 94.44444444444444
